# 02 - CNN Pakcoy: klasifikasi tahap tumbuh dari foto (MobileNetV2, transfer learning)

**Kelas**: `masa_pertumbuhan`, `mendekati_panen`, `siap_panen` (sama dengan folder gambar dan kolom Excel).

**Sebelum mulai**
- Upload `dataset_gambar_pakcoy_monitoring.zip` ke `MyDrive/pakcoy-project/data/` (sudah kamu lakukan).
- Aktifkan GPU: Runtime > Change runtime type > **T4 GPU**.
- Notebook ini otomatis mendeteksi isi zip: kalau sudah ada folder `train/val/test`, dipakai apa adanya; kalau hanya 3 folder kelas, notebook membuang duplikat lalu membagi 70/15/15 sendiri.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, shutil, zipfile, hashlib, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

BASE = '/content/drive/MyDrive/pakcoy-project'
MODELS = f'{BASE}/models'
os.makedirs(MODELS, exist_ok=True)
CLASS_NAMES = ['masa_pertumbuhan', 'mendekati_panen', 'siap_panen']
IMG = (224, 224)
BATCH = 16
SEED = 42
keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Ekstrak zip ke penyimpanan lokal Colab (jauh lebih cepat daripada membaca dari Drive)

In [ ]:
zips = glob.glob(f'{BASE}/data/dataset_gambar*.zip')
print('zip ditemukan:', zips)
assert zips, 'Zip gambar tidak ditemukan di MyDrive/pakcoy-project/data/'
EXTRACT = '/content/img_raw'
shutil.rmtree(EXTRACT, ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(EXTRACT)

def find_root(base):
    for dp, dn, fn in os.walk(base):
        if all(s in dn for s in ['train', 'val', 'test']):
            return dp, 'split'
        if all(c in dn for c in CLASS_NAMES):
            return dp, 'flat'
    raise RuntimeError('Struktur folder tidak dikenali. Harus ada folder train/val/test atau 3 folder kelas.')

ROOT, MODE = find_root(EXTRACT)
print('root:', ROOT, '| struktur:', MODE)

In [ ]:
EXT = ('.jpg', '.jpeg', '.png')

if MODE == 'flat':
    # buang duplikat persis, lalu bagi 70/15/15 per kelas
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = '/content/img_split'
    shutil.rmtree(SPLIT_DIR, ignore_errors=True)
    for c in CLASS_NAMES:
        files = sorted(f for f in glob.glob(f'{ROOT}/{c}/*') if f.lower().endswith(EXT))
        seen, uniq = set(), []
        for f in files:
            h = hashlib.md5(open(f, 'rb').read()).hexdigest()
            if h not in seen:
                seen.add(h); uniq.append(f)
        print(f'{c}: {len(files)} file, {len(uniq)} unik')
        tr, rest = train_test_split(uniq, test_size=0.30, random_state=SEED)
        va, te = train_test_split(rest, test_size=0.50, random_state=SEED)
        for name, lst in [('train', tr), ('val', va), ('test', te)]:
            os.makedirs(f'{SPLIT_DIR}/{name}/{c}', exist_ok=True)
            for f in lst:
                shutil.copy(f, f'{SPLIT_DIR}/{name}/{c}/')
    ROOT = SPLIT_DIR

TRAIN, VAL, TEST = f'{ROOT}/train', f'{ROOT}/val', f'{ROOT}/test'
counts = {}
for s, d in [('train', TRAIN), ('val', VAL), ('test', TEST)]:
    counts[s] = {c: len([f for f in os.listdir(f'{d}/{c}') if f.lower().endswith(EXT)]) for c in CLASS_NAMES}
import pandas as pd
display(pd.DataFrame(counts))

## 2. Dataset dan augmentasi
Data gambar sedikit, jadi augmentasi (flip, rotasi, zoom, geser, kontras, kecerahan) dipasang **di dalam model** dan hanya aktif saat training.

In [ ]:
def load(d, shuffle):
    return keras.utils.image_dataset_from_directory(
        d, labels='inferred', label_mode='int', class_names=CLASS_NAMES,
        image_size=IMG, batch_size=BATCH, shuffle=shuffle, seed=SEED)

train_ds, val_ds, test_ds = load(TRAIN, True), load(VAL, False), load(TEST, False)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE); val_ds = val_ds.prefetch(AUTOTUNE); test_ds = test_ds.prefetch(AUTOTUNE)

n_tr = np.array([counts['train'][c] for c in CLASS_NAMES])
cw = n_tr.sum() / (len(n_tr) * n_tr)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print('class_weight:', class_weight)

# contoh gambar training
plt.figure(figsize=(10, 4))
for imgs, labels in train_ds.take(1):
    for i in range(min(8, len(imgs))):
        plt.subplot(2, 4, i + 1); plt.imshow(imgs[i].numpy().astype('uint8')); plt.title(CLASS_NAMES[int(labels[i])], fontsize=8); plt.axis('off')
plt.show()

## 3. Model: MobileNetV2 (bobot ImageNet) + kepala klasifikasi 3 kelas

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name='augmentasi')

base = keras.applications.MobileNetV2(input_shape=IMG + (3,), include_top=False, weights='imagenet')
base.trainable = False

inp = keras.Input(shape=IMG + (3,))            # masukan: piksel mentah 0-255
x = augment(inp)
x = layers.Rescaling(1. / 127.5, offset=-1)(x) # sesuai kebutuhan MobileNetV2
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inp, out, name='cnn_pakcoy')

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
      keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=class_weight, callbacks=cb, verbose=2)

### Fine-tuning: buka sebagian lapisan atas MobileNetV2 dengan learning rate sangat kecil

In [ ]:
base.trainable = True
for l in base.layers[:-40]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb2 = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
h2 = model.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=class_weight, callbacks=cb2, verbose=2)

acc = h1.history['accuracy'] + h2.history['accuracy']; vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.plot(acc, label='train'); plt.plot(vacc, label='val'); plt.axvline(len(h1.history['accuracy']) - 1, color='gray', ls='--'); plt.legend(); plt.title('Akurasi (garis putus = mulai fine-tuning)'); plt.show()

## 4. Evaluasi pada data uji
**Catatan jujur**: jumlah gambar uji kecil, jadi satu gambar yang salah mengubah akurasi beberapa persen. Laporkan juga confusion matrix, dan sebut keterbatasan ini di laporan.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true, y_pred, paths = [], [], []
for imgs, labels in test_ds:
    p = model.predict(imgs, verbose=0).argmax(axis=1)
    y_true += labels.numpy().tolist(); y_pred += p.tolist()
y_true, y_pred = np.array(y_true), np.array(y_pred)
test_acc = float((y_true == y_pred).mean())
print(f'Akurasi uji = {test_acc:.3f}  ({int((y_true == y_pred).sum())}/{len(y_true)} gambar)')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap='Greens', xticks_rotation=20)
plt.title('Confusion matrix (uji)'); plt.show()
rep = classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=CLASS_NAMES, output_dict=True, zero_division=0)

## 5. Simpan model

In [ ]:
model.save(f'{MODELS}/model_cnn.keras')
cfg = {'tf_version': tf.__version__, 'class_names': CLASS_NAMES, 'img_size': list(IMG),
       'akurasi_uji': test_acc, 'jumlah_gambar_uji': int(len(y_true)),
       'evaluasi': {'Uji': {'akurasi': test_acc, 'cm': cm.tolist(), 'report': rep}}}
with open(f'{MODELS}/config_cnn.json', 'w') as f:
    json.dump(cfg, f, indent=2, default=float)
print(os.listdir(MODELS))
print('Versi TensorFlow untuk requirements.txt:', tf.__version__)

In [ ]:
# uji cepat: prediksi 1 gambar dari folder test (seperti yang akan dilakukan website)
import PIL.Image as Image
f = sorted(glob.glob(f'{TEST}/siap_panen/*'))[0]
im = np.array(Image.open(f).convert('RGB').resize(IMG), dtype='float32')[None]
p = model(im, training=False).numpy()[0]
print(f, '->', {c: round(float(v), 3) for c, v in zip(CLASS_NAMES, p)})